In [9]:
%pip -q install -U transformers accelerate huggingface_hub sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 53.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.24 requires torchvision, which is not installed.
transformer-lens 2.17.0 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.3.5 which is incompatible.


In [1]:
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
# Check current CUDA status
import platform, torch
print("Machine:", platform.machine())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("⚠️ CUDA is not available. See instructions above.")

Machine: x86_64
CUDA available: True
CUDA version: 12.6
GPU device: Tesla T4
GPU count: 1


In [3]:
# Verify CUDA setup
if torch.cuda.is_available():
    print("✅ CUDA is ready!")
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Test a simple CUDA operation
    x = torch.randn(3, 3).cuda()
    print(f"\n✅ Test tensor on GPU: {x.device}")
else:
    print("❌ CUDA is still not available.")
    print("\nTroubleshooting:")
    print("1. In Colab: Make sure GPU runtime is enabled (Runtime → Change runtime type → GPU)")
    print("2. Locally: Check if you have an NVIDIA GPU and CUDA toolkit installed")
    print("3. Try restarting the runtime/kernel after enabling GPU")

✅ CUDA is ready!
Device: Tesla T4
Memory: 15.83 GB

✅ Test tensor on GPU: cuda:0


In [4]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Missing HF_TOKEN in Colab Secrets."

login(token=hf_token)  # stores auth for this runtime
print("HF login OK")

HF login OK


In [9]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

MODEL_ID = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map=None,                 # disable auto-sharding
    dtype=torch.float16,             # fp16 on GPU
).to("cuda")                         # FORCE CUDA

prompt = "Write me a poem about Machine Learning."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Write me a poem about Machine Learning. I'll pay you 1000 dollars.

I have a good poem about Machine Learning.

It is called:

"It is all about the algorithm"

It is a poem about Machine Learning.

The algorithm is the most important part of Machine Learning.

The algorithm
